In [0]:
use dbacademy.jobschedule;

In [0]:
%python
# Auto Loader: Incrementally process new CSV files
# Tracks processed files automatically - only new files will be ingested on subsequent runs

from pyspark.sql.functions import (
    date_format,
    current_timestamp,
    col
)

df = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation", "/Volumes/dbacademy/jobschedule/auto_loader_schema_files") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .option("header", "true") \
    .option("delimiter", ";") \
    .option("inferSchema", "true") \
    .load("/Volumes/dbacademy/jobschedule/data_ingestion/*.csv")

# Clean column names: replace spaces with underscores
for c in df.columns:
    df = df.withColumnRenamed(c, c.replace(" ", "_"))

df = (
    df.withColumn("Source_file_name", col("_metadata.file_path"))
    .withColumn("File_Modification_time", col("_metadata.file_modification_time")) 
    .withColumn("batch_run_date", date_format(current_timestamp(), "dd-MM-yyyy HH:mm:ss"))
)

# Write to Delta table with checkpoint tracking
# trigger(availableNow=True) processes all available files then stops
query = df.writeStream \
    .option("checkpointLocation", "/Volumes/dbacademy/jobschedule/auto_loader_checkpoints") \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .toTable("dbacademy.jobschedule.winequality_red_autoloader_py")

query.awaitTermination()

In [0]:
select count(*) from dbacademy.jobschedule.winequality_red_autoloader_py 

count(*)
11193


In [0]:
select distinct t.Source_file_name from dbacademy.jobschedule.winequality_red_autoloader_py t

Source_file_name
null
/Volumes/dbacademy/jobschedule/data_ingestion/winequality-red7.csv
